<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/_banners/opim5509_banner.svg" width="100%" alt="OPIM 5509 banner"/>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/_energy_track/E3_Advanced_Demand_RNN.ipynb)

# Advanced RNN Topics on the Demand Series
--------------------------------------------------
**Dr. Dave Wanik - University of Connecticut**

The upgrades from the *Advanced RNN Theory* notebook, applied to a real forecast: `Conv1D` + `MaxPooling1D` in front of the LSTM (ConvLSTM), recurrent dropout, stacking, and `Bidirectional`. One bake-off table at the end - because the point is not that these are fancier, it's whether they **beat the baselines and the plain LSTM**.

*Energy-track version of the temperature/occupancy advanced notebooks.*

🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 14B — ConvLSTM, recurrent dropout and Bidirectional on the demand series
- Same tensors as the multivariate RNN notebook - the only thing that changes is the model.
- Conv1D(filters=32, kernel_size=3) on a (24, 8) window -> (22, 32); MaxPooling1D(2) -> (11, 32); the LSTM then spins 11 times, not 24. Read the shapes off summary().
- Recurrent dropout drops the SAME units at every time step; plain dropout would drop different ones. Both are regularizers - one respects the sequence.
- Bidirectional doubles the units (forward + backward). Say the honest caveat: for FORECASTING the backward pass sees nothing the forward pass didn't - the window is all past. It shines in text (Module 5).
- Bake-off table: persistence 129, plain LSTM ~51, ConvLSTM ~52, recurrent-dropout LSTM ~104 (regularizing a model that wasn't overfitting), stacked ~54, bidirectional ~40. The winner is whatever beats persistence by the most - and it is the one you just warned them about, so read it honestly: more units, not magic.
-->


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, classification_report, confusion_matrix
from keras.models import Sequential, load_model
from keras.layers import Dense, Dropout, SimpleRNN, LSTM, GRU, Bidirectional, Conv1D, MaxPooling1D, Flatten
from keras.callbacks import EarlyStopping
import keras
keras.utils.set_random_seed(5509)   # reproducibility: same numbers every run (CPU exact; a GPU may drift a little)

## Read, sort, split

In [ ]:
url = "https://raw.githubusercontent.com/drdave-teaching/OPIM5509Files/main/OPIM5509_Module4_Files/data/BDL_cleanweather_energy.csv"
df = pd.read_csv(url, parse_dates=["Datetime"])
print("in date order as delivered?", df["Datetime"].is_monotonic_increasing)      # it is NOT - always check
df = df.sort_values("Datetime").set_index("Datetime").ffill()

# two years is plenty for the lecture: train on 2018, test on 2019 - chronological, never shuffled
data  = df.loc["2018-01-01":"2019-12-31", ["BDL_tmpf", "BDL_dwpf", "BDL_relh", "Demand"]].copy()
data["hour_sin"] = np.sin(2*np.pi*data.index.hour/24); data["hour_cos"] = np.cos(2*np.pi*data.index.hour/24)
data["dow_sin"]  = np.sin(2*np.pi*data.index.dayofweek/7); data["dow_cos"] = np.cos(2*np.pi*data.index.dayofweek/7)
data = data[["BDL_tmpf", "BDL_dwpf", "BDL_relh", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "Demand"]]   # Demand LAST
train, test = data.loc[:"2018-12-31"], data.loc["2019-01-01":]
print("train:", train.shape, "| test:", test.shape)
data.head()

## Baselines

In [ ]:
y = test["Demand"]
baselines = pd.Series({
    "mean-only":                       mean_absolute_error(y, np.full(len(y), train["Demand"].mean())),
    "seasonal naive (same hour yday)": mean_absolute_error(y.iloc[24:], y.shift(24).iloc[24:]),
    "persistence (last hour)":         mean_absolute_error(y.iloc[1:],  y.shift(1).iloc[1:]),
}, name="test MAE (MW)").round(1)
baselines

## Prep (same as the multivariate RNN notebook)

<!-- WINDOW-FN -->
## The window-making function: `split_sequences`

Target in the **last column**, then:

- **`n_steps`** is the **look-back**: X is the last `n_steps` rows of **every column, the target included**. Yesterday's demand is the most useful input there is.
- y is the target **one step** after the window ends.

Result: X is `(samples, n_steps, n_features)`.


In [ ]:
def split_sequences(seqs, n_steps):
    X, y = [], []
    for i in range(len(seqs) - n_steps):
        X.append(seqs[i:i+n_steps, :]); y.append(seqs[i+n_steps, -1])   # inputs = EVERY column (demand's own past included); target = last col, NEXT step
    return np.array(X), np.array(y)

n_steps = 24
sc   = MinMaxScaler().fit(train)               # fit on TRAIN only
sc_y = MinMaxScaler().fit(train[["Demand"]])   # a separate scaler for the target so we can get MW back
tr_s, te_s = sc.transform(train), sc.transform(test)
X_train, y_train = split_sequences(tr_s, n_steps)
X_test,  y_test  = split_sequences(te_s, n_steps)
n_features = X_train.shape[2]
y_true = sc_y.inverse_transform(y_test.reshape(-1, 1)).ravel()   # test target in MW
print("train tensor:", X_train.shape, "-> (samples, look-back, features) | test:", X_test.shape)

## The plain LSTM to beat

In [ ]:
es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1)

def fit(m, name):
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    m.fit(X_train, y_train, epochs=50, batch_size=64, validation_split=0.2, callbacks=[es], verbose=0)
    pred = sc_y.inverse_transform(m.predict(X_test, verbose=0)).ravel()
    mae = mean_absolute_error(y_true, pred)
    print(f"{name:22s} params: {m.count_params():6d}   test MAE: {mae:.1f} MW")
    return m, pred, mae

plain, pred_plain, mae_plain = fit(Sequential([LSTM(32, input_shape=(n_steps, n_features)), Dense(1)]), "plain LSTM")

## ConvLSTM: `Conv1D` + `MaxPooling1D` in front of the recurrent layer

The convolution slides a kernel of length 3 down the 24-hour window and produces 32 new sequences of length 22; pooling halves that to 11. The LSTM reads a shorter, richer sequence.

In [ ]:
convlstm = Sequential([Conv1D(filters=32, kernel_size=3, activation="relu", input_shape=(n_steps, n_features)),   # (24, 8) -> (22, 32)
                       MaxPooling1D(pool_size=2),                                                                  # (22, 32) -> (11, 32)
                       LSTM(32),
                       Dense(1)])
convlstm.summary()
convlstm, pred_conv, mae_conv = fit(convlstm, "ConvLSTM")

## Recurrent dropout, and stacking

`recurrent_dropout` drops the *same* hidden units at every time step of a sequence, so the regularization respects the order. Plain `dropout` on the inputs drops different features at different steps.

In [ ]:
drop = Sequential([LSTM(32, dropout=0.1, recurrent_dropout=0.2, input_shape=(n_steps, n_features)), Dense(1)])
drop, pred_drop, mae_drop = fit(drop, "LSTM + rec. dropout")

stack = Sequential([LSTM(32, return_sequences=True, input_shape=(n_steps, n_features)), LSTM(16), Dense(1)])
stack, pred_stack, mae_stack = fit(stack, "stacked LSTM")

## Bidirectional

Two LSTMs, one reading the window forwards and one backwards, outputs concatenated - so `Bidirectional(LSTM(32))` has **64** outputs and twice the parameters. For a *forecast* the backward pass can't see anything the forward pass didn't (the window is all past), so don't expect magic here. In Module 5, on text, it matters.

In [ ]:
bi = Sequential([Bidirectional(LSTM(32), input_shape=(n_steps, n_features)), Dense(1)])
bi.summary()
bi, pred_bi, mae_bi = fit(bi, "Bidirectional LSTM")

## The bake-off

In [ ]:
pd.Series({"persistence": baselines.iloc[2], "seasonal naive": baselines.iloc[1],
           "plain LSTM": mae_plain, "ConvLSTM": mae_conv, "LSTM + rec. dropout": mae_drop, "stacked LSTM": mae_stack, "Bidirectional LSTM": mae_bi},
          name="test MAE, 1h ahead (MW)").round(1).sort_values()

In [ ]:
i0 = 24*7*28
plt.figure(figsize=(11, 3)); plt.plot(y_true[i0:i0+24*7], label="actual", lw=2)
plt.plot(pred_plain[i0:i0+24*7], label="plain LSTM"); plt.plot(pred_conv[i0:i0+24*7], label="ConvLSTM")
plt.title("One test week"); plt.ylabel("MW"); plt.legend(); plt.show()

## Save the model and use it again

In [ ]:
convlstm.save('E3_Advanced_Demand_RNN.keras')
reloaded = load_model('E3_Advanced_Demand_RNN.keras')
print("reloaded model reproduces the predictions:", np.allclose(convlstm.predict(X_test[:5], verbose=0), reloaded.predict(X_test[:5], verbose=0)))

## On your own

- Change `kernel_size` to 6 and `pool_size` to 4. Trace the shapes by hand *before* you run `summary()`.
- Put two `Conv1D` layers in front of the LSTM. Does the sequence get too short to be useful?
- Re-run the bake-off with `n_steps = 168`. Which of these architectures benefits from a longer window?